[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/projects/01_regression/project.ipynb)

# Project 01 — Định giá nhà (Regression)

> Một sàn bất động sản muốn **gợi ý giá đăng bán** cho người bán nhà ở TP.HCM. Hiện nay nhân viên ước lượng bằng "giá trung vị mỗi m² của quận × diện tích". Bạn có dữ liệu giao dịch **thô** (`house_prices_dirty.csv`) và cần một mô hình tốt hơn cách làm hiện tại — kèm giải thích được.

**Bạn sẽ làm** (mỗi bước có ô `### START CODE HERE ###` và kiểm tra tự động):
1. Kiểm tra & làm sạch dữ liệu thô (trùng, thiếu, danh mục viết khác nhau, lỗi gõ diện tích).
2. Chia train/test; baseline "trung vị" và baseline **nghiệp vụ** (giá/m² theo quận).
3. Pipeline tiền xử lý + hồi quy tuyến tính trên **log giá** (Ridge, chọn α bằng CV) và gradient boosting; so sánh bằng cross-validation.
4. Đánh giá trên test: MAE, MAPE, R²; phân tích sai số theo quận và theo khoảng giá.
5. Diễn giải: hệ số theo % giá; báo cáo ngắn.

**Tiêu chí đạt**: dữ liệu sạch đúng 5 quận, không dòng trùng, không diện tích vô lý · MAPE test < 16% · MAE test thấp hơn baseline nghiệp vụ ít nhất 15%.

**Chương liên quan**: 03 (EDA), 04 (tiền xử lý, pipeline), 06 (hồi quy tuyến tính, regularization), 12 (gradient boosting), 14 (metric hồi quy), 15 (cross-validation), 17 (log-transform).

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm bài. Bản bài tập (đã xoá lời giải) nằm trong thư mục `notebooks/` của chương.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/projects/01_regression

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.utils import repo_path

raw = pd.read_csv(repo_path("data", "sample", "house_prices_dirty.csv"))
print(raw.shape)
raw.head()

## Bước 1 — Kiểm tra dữ liệu thô

Đếm: số dòng trùng hoàn toàn `n_duplicates`; số giá trị thiếu theo cột `missing` (Series); số cách viết khác nhau của cột `district` `n_district_spellings`. Xem phân phối `area_m2` — có giá trị nào vô lý?

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
n_duplicates = int(raw.duplicated().sum())
missing = raw.isna().sum()
n_district_spellings = raw.district.nunique()
### END CODE HERE ###
print(f"trùng: {n_duplicates} | cách viết quận: {n_district_spellings}")
print("thiếu:", missing[missing > 0].to_dict())
print(raw.area_m2.describe().round(1).to_dict())
print("10 diện tích lớn nhất:", sorted(raw.area_m2)[-10:])

In [ ]:
check_equal(n_duplicates, 15, "số dòng trùng")
check_equal(int(missing.sum()), 234, "tổng số giá trị thiếu")
check_greater(n_district_spellings, 5, "có nhiều cách viết hơn 5 quận thật")

Diện tích 700–1.300 m² cho căn 2–3 phòng ngủ với giá vài tỉ là vô lý — dấu hiệu **gõ thừa một số 0** (diện tích × 10). Đây là lỗi sửa được: chia 10.

## Bước 2 — Làm sạch

Viết `clean(raw)`: bỏ dòng trùng; chuẩn hoá `district` về 5 tên chuẩn (`Quận 1`, `Quận 3`, `Quận 7`, `Bình Thạnh`, `Thủ Đức` — chú ý khoảng trắng, chữ hoa/thường, viết tắt `Q. 1`); sửa `area_m2 > 400` thành `area_m2 / 10`. **Không** điền giá trị thiếu ở đây (để pipeline xử lý — tránh rò rỉ thông tin test).

In [ ]:
def clean(raw: pd.DataFrame) -> pd.DataFrame:
    ### START CODE HERE ### (≈ 5 dòng)
    # STARTER: return raw.copy()
    df = raw.drop_duplicates().copy()
    district = df.district.str.strip().str.lower().str.replace("q. ", "quận ", regex=False)
    df["district"] = district.str.title()
    df.loc[df.area_m2 > 400, "area_m2"] = df.loc[df.area_m2 > 400, "area_m2"] / 10
    return df.reset_index(drop=True)
    ### END CODE HERE ###


houses = clean(raw)
print(houses.shape, houses.district.value_counts().to_dict())

In [ ]:
check_equal(sorted(houses.district.unique()), ["Bình Thạnh", "Quận 1", "Quận 3", "Quận 7", "Thủ Đức"], "đúng 5 quận chuẩn")
check_equal(int(houses.duplicated().sum()), 0, "không còn dòng trùng")
check_less(houses.area_m2.max(), 400, "không còn diện tích vô lý")
check_equal(len(houses), 1500, "1.500 căn sau khi bỏ trùng")
check_true(houses.age_years.isna().any(), "giá trị thiếu được giữ lại cho pipeline")

## Bước 3 — Chia dữ liệu và baseline

Chia 80/20 (`random_state=42`). Hai baseline trên test:
- `mae_median`: luôn dự đoán trung vị giá của train.
- `mae_business`: **cách làm hiện tại** — trung vị (giá/m²) của từng quận tính trên train × diện tích.

In [ ]:
NUMERIC = ["area_m2", "bedrooms", "bathrooms", "age_years", "distance_to_center_km"]
CATEGORICAL = ["district", "has_garden"]
X = houses[NUMERIC + CATEGORICAL]
y = houses.price_billion_vnd
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
### START CODE HERE ### (≈ 4 dòng)
mae_median = mean_absolute_error(y_te, np.full(len(y_te), y_tr.median()))
price_per_m2 = (y_tr / X_tr.area_m2).groupby(X_tr.district).median()
business_pred = X_te.area_m2 * X_te.district.map(price_per_m2)
mae_business = mean_absolute_error(y_te, business_pred)
### END CODE HERE ###
print(f"MAE — trung vị: {mae_median:.3f} tỉ | nghiệp vụ (giá/m² theo quận): {mae_business:.3f} tỉ (MAPE {mean_absolute_percentage_error(y_te, business_pred):.1%})")

In [ ]:
check_greater(mae_median, 1.5, "baseline trung vị rất kém")
check_less(mae_business, 0.8, "baseline nghiệp vụ khá mạnh")

## Bước 4 — Mô hình

Giá nhà lệch phải và tác động của các yếu tố mang tính **nhân** (quận đắt hơn x% chứ không cộng thêm y tỉ) → mô hình hoá **log giá**. Tạo hai mô hình (đều dùng `TransformedTargetRegressor(func=np.log, inverse_func=np.exp)`):
- `ridge`: điền thiếu (median) → `log1p` → chuẩn hoá cho biến số; one-hot cho `district`, `has_garden`; `RidgeCV(alphas=np.logspace(-3, 3, 13))`.
- `boosting`: biến số giữ nguyên, one-hot (dense) cho biến phân loại; `HistGradientBoostingRegressor(random_state=0)` (tự xử lý giá trị thiếu).

So sánh bằng 5-fold CV (MAE) trên train → `cv_results` (dict tên → (trung bình, độ lệch chuẩn)).

In [ ]:
### START CODE HERE ### (≈ 12 dòng)
# STARTER: ridge, boosting, cv_results = None, None, {}
linear_pre = ColumnTransformer([("num", make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()), NUMERIC),
                                ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)])
ridge = TransformedTargetRegressor(make_pipeline(linear_pre, RidgeCV(alphas=np.logspace(-3, 3, 13))), func=np.log, inverse_func=np.exp)
tree_pre = ColumnTransformer([("num", "passthrough", NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL)])
boosting = TransformedTargetRegressor(make_pipeline(tree_pre, HistGradientBoostingRegressor(random_state=0)), func=np.log, inverse_func=np.exp)
cv_results = {}
for name, model in [("ridge", ridge), ("boosting", boosting)]:
    scores = -cross_val_score(model, X_tr, y_tr, cv=KFold(5, shuffle=True, random_state=0), scoring="neg_mean_absolute_error")
    cv_results[name] = (scores.mean(), scores.std())
### END CODE HERE ###
for name, (m, s) in cv_results.items():
    print(f"{name:9s}: CV MAE = {m:.3f} ± {s:.3f}")

In [ ]:
check_equal(sorted(cv_results), ["boosting", "ridge"], "hai mô hình")
check_less(cv_results["ridge"][0], 0.6, "ridge trên log giá")
check_less(abs(cv_results["ridge"][0] - cv_results["boosting"][0]), 2 * max(cv_results["ridge"][1], cv_results["boosting"][1]), "hai mô hình chênh nhau trong phạm vi nhiễu CV")

Hai mô hình gần như ngang nhau (chênh lệch nhỏ hơn độ lệch chuẩn giữa các fold). Chọn **Ridge**: đơn giản, nhanh, và **diễn giải được** — điều quan trọng khi nhân viên phải giải thích giá gợi ý cho khách.

## Bước 5 — Đánh giá trên test (một lần)

Huấn luyện `ridge` trên toàn bộ train, dự đoán test → `pred_test`; tính `mae_test`, `mape_test`, `r2_test`.

In [ ]:
### START CODE HERE ### (≈ 4 dòng)
pred_test = ridge.fit(X_tr, y_tr).predict(X_te)
mae_test = mean_absolute_error(y_te, pred_test)
mape_test = mean_absolute_percentage_error(y_te, pred_test)
r2_test = r2_score(y_te, pred_test)
### END CODE HERE ###
print(f"TEST: MAE = {mae_test:.3f} tỉ | MAPE = {mape_test:.1%} | R² = {r2_test:.3f} | cải thiện so với baseline nghiệp vụ: {1 - mae_test / mae_business:.0%}")

In [ ]:
check_less(mape_test, 0.16, "MAPE test < 16%")
check_less(mae_test, 0.85 * mae_business, "tốt hơn cách làm hiện tại ít nhất 15%")
check_greater(r2_test, 0.8, "R² test")

## Bước 6 — Phân tích sai số

Sai số trung bình **theo quận** và **theo ngũ phân vị giá thật** (`error_by_district`, `error_by_price` — DataFrame với cột `MAE` và `MAPE`). Mô hình sai nhiều ở đâu?

In [ ]:
### START CODE HERE ### (≈ 5 dòng)
errors = pd.DataFrame({"district": X_te.district.values, "price": y_te.values, "abs_err": np.abs(y_te.values - pred_test)})
errors["pct_err"] = errors.abs_err / errors.price
error_by_district = errors.groupby("district").agg(MAE=("abs_err", "mean"), MAPE=("pct_err", "mean"))
error_by_price = errors.groupby(pd.qcut(errors.price, 5), observed=True).agg(MAE=("abs_err", "mean"), MAPE=("pct_err", "mean"))
### END CODE HERE ###
print(error_by_district.round(3), "\n")
print(error_by_price.round(3))

In [ ]:
check_equal(list(error_by_district.columns), ["MAE", "MAPE"], "cột MAE, MAPE")
check_equal(len(error_by_price), 5, "5 nhóm giá")
check_greater(error_by_price.MAE.iloc[-1], error_by_price.MAE.iloc[0], "sai số tuyệt đối tăng theo giá")

**Nhận xét mẫu:** Sai số **tuyệt đối** tăng mạnh theo giá (nhà đắt sai nhiều tỉ hơn) nhưng sai số **phần trăm** khá đều — đúng bản chất mô hình trên log giá (sai số nhân). Vì vậy báo cáo cho khách nên dùng **khoảng giá theo %** (ví dụ ±15%) thay vì ± một số tỉ cố định. Quận 1 có sai số tuyệt đối lớn nhất (giá cao nhất) dù sai số % tương đương các quận khác. Muốn giảm thêm cần đặc trưng vị trí chi tiết (phường, mặt tiền, hẻm) và nhiều dữ liệu hơn ở phân khúc giá cao.

## Bước 7 — Diễn giải

Với mô hình log, hệ số $\beta$ của biến one-hot nghĩa là giá nhân với $e^{\beta}$. Lấy hệ số của các quận và `has_garden` từ Ridge, đổi sang **% chênh lệch giá** → Series `effects_pct` (index = tên đặc trưng sau one-hot, ví dụ `cat__district_Quận 1`).

In [ ]:
### START CODE HERE ### (≈ 3 dòng)
fitted = ridge.regressor_
names = fitted[0].get_feature_names_out()
effects_pct = pd.Series((np.exp(fitted[-1].coef_) - 1) * 100, index=names).filter(like="cat__")
### END CODE HERE ###
print(effects_pct.round(1).sort_values(ascending=False).to_string())

In [ ]:
check_greater(effects_pct["cat__district_Quận 1"], effects_pct["cat__district_Thủ Đức"] + 50, "Quận 1 đắt hơn Thủ Đức rất nhiều (cùng diện tích, tuổi...)")
check_greater(effects_pct["cat__has_garden_True"], effects_pct["cat__has_garden_False"], "có sân vườn → giá cao hơn")

Lưu ý khi đọc: one-hot không bỏ cột nào nên mỗi hệ số là chênh lệch so với một "quận trung bình". Để so hai quận, chia hai hệ số nhân: ví dụ Quận 1 so với Thủ Đức là $\frac{1 + 0.509}{1 - 0.313} - 1 \approx +120\%$ cho cùng một căn nhà.

## Bước 8 — Báo cáo (viết bằng lời của bạn)

Viết 5–8 câu cho trưởng phòng kinh doanh: mô hình tốt hơn cách làm hiện tại bao nhiêu (MAE, MAPE, kèm CV ± độ lệch chuẩn); nên trình bày giá gợi ý thế nào (khoảng %); hạn chế (dữ liệu chỉ 5 quận, thiếu vị trí chi tiết, dữ liệu một thời điểm — giá thay đổi theo thời gian); bước tiếp theo.

✍️ Báo cáo:

**Báo cáo mẫu:** Mô hình hồi quy Ridge trên log giá dự đoán giá với sai số trung bình ~0.50 tỉ (MAPE ~11%) trên tập kiểm tra, tốt hơn cách ước lượng hiện tại (giá/m² trung vị theo quận, MAE ~0.65 tỉ, MAPE ~14%) khoảng 23%; kết quả cross-validation ổn định (độ lệch chuẩn ~0.04 tỉ giữa các fold). Gradient boosting không tốt hơn đáng kể nên chọn mô hình tuyến tính vì dễ giải thích: mỗi quận, tuổi nhà, sân vườn tác động theo **%** giá. Đề xuất hiển thị giá gợi ý dưới dạng khoảng ±15% quanh dự đoán. Hạn chế: chỉ có 5 quận, thiếu đặc trưng vị trí chi tiết (phường, mặt tiền) và thời điểm giao dịch — cần huấn luyện lại định kỳ khi thị trường biến động. Bước tiếp theo: thêm dữ liệu vị trí, khoảng dự đoán bằng quantile regression hoặc conformal prediction (chương 37), và giám sát sai số khi có giao dịch mới.